# Can we predict 12-month loan default?
**Target:** `defaulted` (1 = defaulted within 12 months after funding).
**Plan:** inspect data → check for leakage/quality issues → baselines → logistic regression and gradient boosting with cross-validation → one-time evaluation on a held-out test set → calibration, threshold and feature-importance analysis → conclusion.

Metrics: with a ~10% positive rate, accuracy is misleading (predicting "no default" always gives ~90%). We use ROC-AUC and PR-AUC (average precision; the no-skill baseline equals the default rate).

In [ ]:
import os, warnings
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.environ.get('TMPDIR','/tmp'), 'mpl'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             brier_score_loss, confusion_matrix)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=42
plt.rcParams.update({'figure.dpi':100,'axes.spines.top':False,'axes.spines.right':False})

## 1. Data and quality checks

In [ ]:
df = pd.read_csv('data.csv')
print(df.shape)
display(pd.read_csv('columns.csv'))
display(df.head())
print('missing values:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()),
      '| duplicate IDs:', int(df.application_id.duplicated().sum()))
display(df.describe().T)

In [ ]:
rate = df.defaulted.mean()
print(f'Default rate: {rate:.2%}  ({df.defaulted.sum()} of {len(df)})')
print('Majority-class accuracy:', f'{1-rate:.2%}')

Data is clean (no missing values or duplicates). Classes are imbalanced (~10% defaults).
`application_id` is an identifier, so it is **excluded from features** (it carries no causal signal and could only add noise or leak ordering). There is no date column, so we cannot do a time-based split; we use stratified random splits. All listed features are measured *at application*, so none leaks the outcome.

## 2. Exploratory analysis

In [ ]:
num = ['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig, axes = plt.subplots(2,3, figsize=(13,7))
for ax,c in zip(axes.ravel(), num):
    q = pd.qcut(df[c], 10, duplicates='drop')
    g = df.groupby(q, observed=True).defaulted.mean()
    ax.bar(range(len(g)), g.values, color='#3b6fb6')
    ax.axhline(rate, color='k', ls='--', lw=1)
    ax.set_title(f'Default rate by {c} decile'); ax.set_xlabel('decile (low → high)'); ax.set_ylabel('default rate')
plt.tight_layout(); plt.show()
display(df.groupby('region').defaulted.agg(default_rate='mean', n='size'))

In [ ]:
print(df[num+['defaulted']].corr(method='spearman')['defaulted'].drop('defaulted').sort_values())

Debt ratio shows the clearest positive relationship; income and employment tenure are negative; larger loans are riskier. Region, age and credit lines look like noise at this level. We'll let the models and a held-out set tell us how much of this is real.

## 3. Train / test split
The test set (20%) is touched **once**, at the end. Model selection uses cross-validation on the training set only.

In [ ]:
features = ['region']+num
X, y = df[features], df.defaulted
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(X_tr.shape, X_te.shape, f'train rate {y_tr.mean():.3f}, test rate {y_te.mean():.3f}')

## 4. Models (5-fold stratified CV on training data)

In [ ]:
pre = ColumnTransformer([('num', StandardScaler(), num),
                         ('cat', OneHotEncoder(drop='first'), ['region'])])
models = {
 'Baseline (prior)': DummyClassifier(strategy='prior'),
 'Logistic regression': Pipeline([('pre',pre),('m',LogisticRegression(max_iter=1000))]),
 'Logistic regression (balanced)': Pipeline([('pre',pre),('m',LogisticRegression(max_iter=1000, class_weight='balanced'))]),
 'Gradient boosting': Pipeline([('pre',ColumnTransformer([('num','passthrough',num),('cat',OneHotEncoder(drop='first'),['region'])])),
                                ('m',HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                      min_samples_leaf=40, l2_regularization=1.0, random_state=SEED))]),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows=[]
for name,m in models.items():
    s = cross_validate(m, X_tr, y_tr, cv=cv, scoring={'roc_auc':'roc_auc','pr_auc':'average_precision','brier':'neg_brier_score'})
    rows.append({'model':name,'ROC-AUC':s['test_roc_auc'].mean(),'AUC std':s['test_roc_auc'].std(),
                 'PR-AUC':s['test_pr_auc'].mean(),'Brier':-s['test_brier'].mean()})
cvres = pd.DataFrame(rows).set_index('model').round(4)
display(cvres)

Logistic regression and gradient boosting are compared on cross-validated scores. If boosting does not clearly beat the simpler linear model, the signal is largely additive and the linear model is preferable for interpretability.

## 5. Final evaluation on the held-out test set

In [ ]:
final_name = cvres.drop('Baseline (prior)')['ROC-AUC'].idxmax()
print('Selected by CV ROC-AUC:', final_name)
final = models[final_name].fit(X_tr, y_tr)
p = final.predict_proba(X_te)[:,1]
yt = y_te.values
print(f'Test ROC-AUC: {roc_auc_score(yt,p):.3f}')
print(f'Test PR-AUC : {average_precision_score(yt,p):.3f}  (no-skill = {yt.mean():.3f})')
print(f'Test Brier  : {brier_score_loss(yt,p):.4f}  (prior-only = {brier_score_loss(yt, np.full(len(yt), y_tr.mean())):.4f})')

# bootstrap CI for ROC-AUC
rng = np.random.default_rng(SEED)
aucs = [roc_auc_score(yt[i], p[i]) for i in (rng.integers(0, len(yt), len(yt)) for _ in range(1000))]
print('ROC-AUC 95%% CI: %.3f - %.3f' % tuple(np.percentile(aucs, [2.5, 97.5])))

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,4.5))
fpr,tpr,_ = roc_curve(yt,p); ax[0].plot(fpr,tpr,color='#3b6fb6'); ax[0].plot([0,1],[0,1],'k--',lw=1)
ax[0].set(title='ROC curve (test)', xlabel='False positive rate', ylabel='True positive rate')
pr,rc,_ = precision_recall_curve(yt,p); ax[1].plot(rc,pr,color='#3b6fb6'); ax[1].axhline(yt.mean(),color='k',ls='--',lw=1)
ax[1].set(title='Precision-recall (test)', xlabel='Recall', ylabel='Precision')
fo,mp = calibration_curve(yt,p,n_bins=8,strategy='quantile'); ax[2].plot(mp,fo,'o-',color='#3b6fb6'); ax[2].plot([0,.4],[0,.4],'k--',lw=1)
ax[2].set(title='Calibration (test)', xlabel='Mean predicted probability', ylabel='Observed default rate')
plt.tight_layout(); plt.show()

**Note:** if the selected model is the class-balanced variant, its probabilities are inflated by design, so the calibration plot will sit below the diagonal. Ranking metrics (AUC, lift) are unaffected.

### Practical value: risk ranking

In [ ]:
t = pd.DataFrame({'p':p,'y':yt})
t['risk_decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=range(1,11))
lift = t.groupby('risk_decile', observed=True).y.agg(default_rate='mean', n='size')
lift['lift_vs_avg'] = lift.default_rate / t.y.mean()
display(lift.round(3))
top = t.nlargest(int(0.2*len(t)),'p').y.sum()/t.y.sum()
print(f'Top 20% riskiest applicants capture {top:.0%} of all defaults (random = 20%).')

In [ ]:
# Threshold trade-offs (quantile-based so they work for any probability scale)
rows=[]
for q in [0.05,0.10,0.20,0.30]:
    thr = np.quantile(p, 1-q)
    pred=(p>=thr).astype(int); tn,fp,fn,tp=confusion_matrix(yt,pred).ravel()
    rows.append({'flag top %':q*100,'threshold':thr,'precision':tp/max(tp+fp,1),'recall':tp/(tp+fn)})
display(pd.DataFrame(rows).set_index('flag top %').round(3))

The right threshold depends on the cost of a missed default versus a declined good loan, which isn't given in the data, so this table is illustrative rather than a recommendation.

## 6. What drives the predictions?

In [ ]:
pi = permutation_importance(final, X_te, y_te, scoring='roc_auc', n_repeats=30, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=features); err = pd.Series(pi.importances_std, index=features)
imp = imp.sort_values()
plt.barh(imp.index, imp.values, xerr=err[imp.index], color='#3b6fb6'); plt.xlabel('Drop in test ROC-AUC when shuffled')
plt.title('Permutation importance'); plt.show()

In [ ]:
if final_name.startswith('Logistic'):
    names = num + ['region_' + c for c in final.named_steps['pre'].named_transformers_['cat'].categories_[0][1:]]
    coef = pd.Series(final.named_steps['m'].coef_[0], index=names)
    print('Standardised coefficients (log-odds per 1 SD; region vs. baseline region):')
    display(coef.sort_values().round(3).to_frame('coef'))

## 7. Conclusion

In [ ]:
from IPython.display import Markdown
auc = roc_auc_score(yt,p)
display(Markdown('**Result:** the selected model (*%s*) reaches test ROC-AUC **%.2f** and PR-AUC **%.2f** versus a no-skill %.2f; the top 20%% highest-risk applicants contain **%.0f%%** of defaults.'
                 % (final_name, auc, average_precision_score(yt,p), yt.mean(), top*100)))

**How to read this (check against the numbers above)**
- Defaults are *partly* predictable from application data: the model ranks risk better than chance, driven mainly by debt ratio, income, employment tenure and loan size. That makes it useful for **prioritising / pricing risk**, but not for identifying individual defaulters with confidence: at useful recall, precision stays low.
- If gradient boosting gives little or no gain over logistic regression, the signal is largely additive and the remainder is noise or information not in these columns.
- Region, age and credit lines add little.

**Caveats**
- Only ~840 defaults, so estimates carry real uncertainty (see the bootstrap CI).
- No time field: stability across time/vintages is untested. Random splits can overstate performance if the population drifts.
- `defaulted` is defined *after funding*, so declined applicants have no outcome. The model reflects past approval policy (selection bias) and may not hold for the full applicant pool.
- Recalibrate and monitor probabilities before using them for decisions; run fairness checks on age and region before any lending use.

**Next steps:** add bureau/behavioural features, validate on a later time period, and choose a threshold from actual cost figures.